In [1]:
from importlib import reload
import torch
import numpy as np
import time
torch.set_default_dtype(torch.float64)
np.random.seed(2)
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
print(device)
import numpy as np
import sys
sys.path.append(r"../main_code/3d")
import generate_data3,grid_cell,visual,cal_S_grad,org_m,mea_3D,net_3D_fix,S_valgrad,error_general_3D
CUDA_LAUNCH_BLOCKING=1

def ana_S(x, center,R, r, S0):
    x1=x[:,0]
    x2=x[:,1]
    x3=x[:,2]
    cx, cy, cz = center[0],center[1],center[2]
    x_rel, y_rel, z_rel = x1 - cx, x2 - cy, x3 - cz
    condition = (np.sqrt(x_rel**2 + y_rel**2) - R)**2 + z_rel**2 <= r**2
    return np.where(condition, S0, 0.0)


cuda:3


## 81 

In [2]:
import copy
kk=[1,5,9,13,17,21,25,29,33,37,41,45,49,53,57,61,65,69,73,77,81]
R_true = 0.25            # 主半径 R
r_true = 0.15            # 次半径 r
S0= 1.0  
tau_x_min,tau_x_max,tau_y_min,tau_y_max,tau_z_min,tau_z_max=-0.5,0.5,-0.5,0.5,-0.5,0.5
b_x_min,b_x_max,b_y_min,b_y_max,b_z_min,b_z_max=-0.75,0.75,-0.75,0.75,-0.75,0.75 #边界gamma处
num_batches_gauss=1
b_n=10
points_b=mea_3D.generate_cube_surface(b_x_min,b_x_max,b_y_min,b_y_max,b_z_min,b_z_max,b_n) #在边界Γ处收集数据
center_true=np.array([0.0,0.0,0.0])
number_gene=100
eps=0.05
batch_number_rec_mea,mea=2,1
cupy_device=3
num_batches_gauss=4
num_batches_appr=10
num_batches_mea=1
condition="Dirichlet"
S_int_true1, F_mea_b1 = generate_data3.boundary_data_torus(kk, number_gene, points_b, ana_S,tau_x_min,tau_x_max,tau_y_min,tau_y_max,tau_z_min,tau_z_max,center_true,R_true, r_true,S0,eps,mea,num_batches_mea,condition,device,cupy_device)

In [6]:
F_diri_5=org_m.F(F_mea_b1,[],[],[],condition)

In [7]:
F_diri_5

array([[ 1.75870475e-03],
       [ 2.90621366e-03],
       [ 3.30792063e-03],
       ...,
       [-9.07771764e-06],
       [-3.99922811e-05],
       [-3.58986209e-05]])

In [8]:
np.save("./noise=5%/F_diri_5%.npy",F_diri_5)